# 11 · Logistic regression for future prostate IHC (NE-positive, NEPC-like, AR-low) on the risk score

Does the frozen federated (no-DFCI) **platinum-trained XGBoost** score, at each ADT landmark, predict that a future
prostate-relevant pathology specimen has an immunohistochemistry (IHC) workup, and whether that IHC is
neuroendocrine (NE)-positive, NEPC-like or AR-low? Each outcome is an unpenalized logistic regression on the score
alone (`outcome ~ RISK_Z`, z-scored within each fit). All logic lives in `COMPASS/multivariable_models/ihc_lr.py`
and `COMPASS/prostate_ihc/annotations.py`.

| Outcome | Event (future prostate-relevant stained specimen) | Population (per landmark) |
|---|---|---|
| `ihc_any` | ≥1 lesional stain row (Ki-67-only workups count) | every at-risk patient |
| `ne_pos` | synaptophysin, chromogranin, INSM1 or CD56 called positive (a discordant NE marker counts as positive) | at-risk patients with no NE-positive specimen at or before ADT + landmark |
| `nepc_like` | NE-positive, ≥1 AR-axis marker negative and none positive | same as `ne_pos` |
| `ar_low` | ≥1 AR-axis marker (AR, PSA, NKX3.1, PSAP, PSMA, prostein) negative and none positive, whatever the NE stains show | at-risk patients with no AR-low specimen at or before ADT + landmark |

**At risk.** The `endpoint == "nepc"` rows of 04's `transfer_scores.csv` (XGBoost, `config == "both"`). Patients
with NEPC or censoring at or before the landmark are not scored there, so they are not in any fit. With
`ONCOPANEL_ONLY` (default), only scored patients in the prostate OncoPanel cohort of the original step-2 output
(`results/oncopanel_prostate_specimen_ihc.csv`, one row per sequenced prostate specimen; only `MRN_INT` is read)
are analysed. Their events can still come from any prostate-relevant specimen, sequenced or not.

**IHC source.** Annotations are rebuilt from the saved step-1 intermediates from the prostate_ihc script (`ihc_extraction/intermediate/`); nothing is re-extracted or re-parsed and no raw data are read
(summary: `notebooks/testing/11_prostate_ihc_extraction.ipynb`):
- `prostate_ihc_long.parquet`: stain mentions. Marker calls are recomputed per specimen (pathology part: MRN,
  accession, part) from its lesional rows; a marker with both positive and negative mentions is *discordant* and
  counts as positive for NE markers and as neither for AR-axis markers.
- `prostate_ihc_specimen_all.parquet`: `prostate_relevant` (evidence tiers 1–4: prostate-lineage IHC positive,
  prostate site, pathologist states prostate origin, NE tumour of unstated origin). Only these specimens count,
  sequenced or not. Its NE+/AR-axis+ flags must agree with the recomputed calls (except at discordant markers), or loading stops.
- `prostate_cohort.parquet`: registry C61 or malignant prostate pathology (plus prostate OncoPanel patients, if built by the current extraction code); scored patients outside it cannot have events (reported below).

No LLM findings are used. `MRN_INT` is `DFCI_MRN`, the scores' `person_id`.

**Timing.** A specimen is dated by its earliest lesional `specimen_date` (pathology outside-specimen date, else
specimen date). "Future" means ADT + landmark < date ≤ ADT + 3650 d. The label is "ever in that window", not a
fixed horizon. Prior IHC excludes no one. A stained specimen with no date is never an event and is counted in
`n_undated_ihc_specimen`. Every analysed patient must have an ADT anchor; missing anchors stop the analysis.

**Statistics.**
- The notebook reports AUCs only: the raw-score AUC (preferred, below) and the out-of-fold AUC (stratified
  5-fold), each with a patient-bootstrap 95% CI. The OOF CI resamples the fixed OOF predictions (folds are not refit).
- ORs (per SD), Wald CIs and LRT p from the full-population fit are written to `lr_coefficients.csv`, not displayed.
- Fewer than `MIN_EVENTS` events or non-events gives a count-only row.

**Raw-score AUC.** `auc_raw_score` is the AUC of the untransformed frozen score itself, with no fitting,
no CV and no sign flip, and a patient-bootstrap 95% CI drawn with the same seed (so the resamples are paired with
the OOF CI). It does not depend on CV, so failed folds never suppress it. For this external, frozen score it is the
preferred discrimination measure. With few events and a weak or null predictor, the OOF AUC is biased below 0.5:
each fold's refitted intercept shifts that fold's predictions, and those shifts can dominate the ranking.
The risk-only LR is monotone in the score, so its full-population AUC equals the raw-score AUC whenever OR > 1.

**Out-of-fold predictions.** `lr_oof_predictions.csv` holds one probability per patient for each outcome × landmark,
covering every patient in that outcome's population. Each probability comes from the CV fit that held that patient
out, the same predictions the AUC uses. `event` is that outcome's label. Each outcome × landmark is its own fit, so
do not pool rows across them. Calibration is not displayed.

**Outputs.** These go under the protected data root `DATA_ROOT/ihc_extraction/lr/` (overwriting the earlier
OncoPanel-CSV outputs):
- aggregate CSVs;
- `lr_oof_predictions.csv` (patient level);
- a specimen-level audit parquet (flags and dates only);
- `run_meta.json`.

**Prerequisites.** Notebook 04 has written `transfer_scores.csv`. The three step-1 intermediates are in
`ihc_extraction/intermediate/`, and (with `ONCOPANEL_ONLY`) `oncopanel_prostate_specimen_ihc.csv` is in
`ihc_extraction/results/`.

In [ ]:
import json, os, sys
sys.path.insert(0, "../../../pipeline")
import compass_pipeline as cp
for _p in (str(cp.PROJECT_ROOT),):
    if _p not in sys.path: sys.path.insert(0, _p)
from pathlib import Path
import pandas as pd
from IPython.display import display
from COMPASS.multivariable_models import ihc_lr as il
from COMPASS.prostate_ihc import annotations as ann
from COMPASS.prostate_ihc import paths as ihc_paths
nsl = il.nsl

DATA_ROOT  = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))
IHC_PATHS  = ihc_paths.default_paths()       # env override: COMPASS_IHC_EXTRACTION_ROOT

# >>> Edit <<<
ONCOPANEL_ONLY = True           # restrict to the prostate OncoPanel patients of the original step-2 output
COHORTS     = ("all",)          # "all" = every scored patient (OncoPanel patients if ONCOPANEL_ONLY)
LANDMARKS   = (0, 90, 180)
MIN_EVENTS  = 5
N_FOLDS     = 5
N_BOOTSTRAP = 200
SEED        = 0

SCORES_PATH   = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"
DATA_PATH     = DATA_ROOT / "longitudinal_prediction_data_adt.csv"      # ADT anchor dates
INTER_DIR     = IHC_PATHS.intermediate_dir                              # pre-existing step-1 parquets
ONCOPANEL_PATH = IHC_PATHS.results_dir / ihc_paths.ONCOPANEL_SPECIMEN_FILE  # step-2 CSV; only MRN_INT is read
OUT_DIR       = IHC_PATHS.intermediate_dir.parent / "lr"   # protected data root; aggregate tables + specimen audit

## Load inputs

In [ ]:
scores = nsl.load_transfer_scores(SCORES_PATH, family="xgboost_cox", config="both", endpoint="nepc")
scores = scores[scores["landmark_days"].isin(LANDMARKS)]
n_scored_total = scores[il.ID_COL].nunique()
oncopanel_ids = il.load_oncopanel_patients(ONCOPANEL_PATH) if ONCOPANEL_ONLY else None
if ONCOPANEL_ONLY:
    scores = scores[scores[il.ID_COL].isin(oncopanel_ids)]
    print(f"OncoPanel restriction: {len(oncopanel_ids):,} prostate OncoPanel patients; "
          f"{scores[il.ID_COL].nunique():,} of {n_scored_total:,} scored patients kept")
anchors = il.load_anchor_dates(DATA_PATH)
members = nsl.cohort_members({c: None if c == "all" else cp.cohort_mrn_list_path(c) for c in COHORTS})
scored_ids = set(scores[il.ID_COL].astype(int))
print(f"{len(scored_ids):,} scored patients; landmarks {sorted(scores['landmark_days'].unique())}")
il.validate_anchor_coverage(scores, anchors)

inter = ann.load_intermediates(INTER_DIR)
calls = ann.marker_calls(inter["long"])
specimens = il.load_ihc_specimens(inter["long"], inter["spec"])
ihc_cohort = set(pd.to_numeric(inter["cohort"]["MRN_INT"]).astype(int))
print(f"{len(scored_ids - ihc_cohort):,} scored patients are not in the prostate_ihc cohort (cannot have events)")

scored_spec = il.add_days_from_adt(specimens[specimens[il.LOCAL_ID_COL].isin(scored_ids)], anchors)
relevant = scored_spec[scored_spec["prostate_relevant"]]
print(f"{len(specimens):,} stained specimens ({specimens[il.LOCAL_ID_COL].nunique():,} patients); "
      f"{len(scored_spec):,} belong to scored patients, {len(relevant):,} of them prostate-relevant "
      f"({relevant[il.LOCAL_ID_COL].nunique():,} patients)")
for label, col in (("with IHC", "ihc_any"), ("NE-positive", "ne_pos"), ("NEPC-like", "nepc_like"), ("AR-low", "ar_low")):
    print(f"  {label:<12} {int(specimens.loc[specimens['prostate_relevant'], col].sum()):>6,} relevant · "
          f"{int(relevant[col].sum()):>6,} relevant, scored")
print(f"  undated (relevant, scored): {int(relevant['specimen_date'].isna().sum()):,}")
display(ann.summary(specimens.rename(columns={il.LOCAL_ID_COL: "MRN_INT"})))

### Marker frequency

This shows, per marker, how many prostate-relevant specimens were stained, how many were called positive, negative or discordant, and how many of the NE-positive specimens it was stained on.

In [ ]:
display(il.marker_frequency(calls, specimens))

## LR frame and attrition

`n_prior_ne_pos` counts patients excluded from both NE fits and `n_prior_ar_low` those excluded from the AR-low fit.
`n_undated_ihc_specimen` counts patients with a prostate-relevant stained specimen that has no date (never an event).

In [ ]:
frame = il.build_ihc_frame(scores, specimens, LANDMARKS, anchors=anchors)
models, coefficients, counts, predictions = il.run_ihc_lr(
    frame, members, LANDMARKS, min_events=MIN_EVENTS, n_folds=N_FOLDS,
    n_bootstrap=N_BOOTSTRAP, seed=SEED, return_predictions=True)
display(counts.set_index(["cohort", "landmark_days"]).T)

## Write outputs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, df in (("lr_models", models), ("lr_coefficients", coefficients), ("lr_counts", counts),
                 ("lr_oof_predictions", predictions)):
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
scored_spec.reindex(columns=il.AUDIT_COLUMNS).to_parquet(OUT_DIR / "ihc_specimen_audit.parquet", index=False)
meta = {
    "ihc_source_version": il.IHC_SOURCE_VERSION,
    "intermediate_dir": str(INTER_DIR),
    "specimen": "pathology part (MRN, ACCESSION, PART) with >=1 lesional stain row; prostate_relevant (tiers 1-4) only",
    "calls": "recomputed from prostate_ihc_long lesional rows; discordant NE = positive, discordant AR-axis = neither",
    "ar_low": ">=1 AR-axis marker negative, none positive, regardless of NE",
    "event_date": "earliest lesional specimen_date per specimen",
    "prior_rule": ("prior NE-positive specimen excludes from ne_pos and nepc_like; prior AR-low excludes from "
                   "ar_low; prior IHC excludes no one"),
    "population": ("scored patients in the prostate OncoPanel cohort (step-2 specimen CSV)" if ONCOPANEL_ONLY
                   else "every scored patient"),
    "oncopanel_path": str(ONCOPANEL_PATH) if ONCOPANEL_ONLY else None,
    "n_oncopanel_patients": len(oncopanel_ids) if ONCOPANEL_ONLY else None,
    "n_scored_total": int(n_scored_total), "n_scored_analysed": len(scored_ids),
    "anchor_policy": "require_all_analysed_patients",
    "scores_path": str(SCORES_PATH), "anchor_path": str(DATA_PATH),
    "cohorts": list(COHORTS), "landmarks": list(LANDMARKS), "max_followup_days": il.MAX_FOLLOWUP_DAYS,
    "oof_predictions": "risk-only LR, stratified K-fold, one row per patient x landmark x outcome",
    "min_events": MIN_EVENTS, "n_folds": N_FOLDS, "n_bootstrap": N_BOOTSTRAP, "seed": SEED,
}
(OUT_DIR / "run_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Wrote lr_models.csv, lr_coefficients.csv, lr_counts.csv, lr_oof_predictions.csv "
      f"({len(predictions):,} patient rows), ihc_specimen_audit.parquet, run_meta.json to {OUT_DIR}")

## Results

These show the out-of-fold AUC and the raw-score AUC (95% CI) per landmark × outcome, with n and events. The ORs,
coefficients and OOF predictions are still written to `OUT_DIR` but not displayed.

In [ ]:
def fmt(est, lo, hi, digits=3):
    return pd.Series([f"{e:.{digits}f} ({l:.{digits}f}–{h:.{digits}f})" if pd.notna(e) else ""
                      for e, l, h in zip(est, lo, hi)], index=est.index)

idx = ["cohort", "landmark_days"]
cols = list(il.OUTCOMES)
mo = models.assign(AUC=lambda d: fmt(d["auc"], d["auc_lo"], d["auc_hi"]),
                   AUC_raw=lambda d: fmt(d["auc_raw_score"], d["auc_raw_score_lo"], d["auc_raw_score_hi"]))
print("=== AUC of the raw risk score (95% CI), no fitting ===")
display(mo.pivot_table(index=idx, columns="outcome", values="AUC_raw", aggfunc="first").reindex(columns=cols))
print("=== Out-of-fold AUC (95% CI) ===")
display(mo.pivot_table(index=idx, columns="outcome", values="AUC", aggfunc="first").reindex(columns=cols))
print("=== n and events ===")
display(mo.pivot_table(index=idx, columns="outcome", values=["n", "events"], aggfunc="first"))

model_notes = models[models["note"].fillna("").ne("")]
if len(model_notes):
    print("\nModels with notes (not fit, separation, or CV failure):")
    display(model_notes[["cohort", "landmark_days", "outcome", "n", "events", "n_failed_folds", "note"]])